# Bangladesh: 0239_sartram

Static adaptation only; Earth Engine/data/asset execution not verified.

Original scientific assumptions and date windows retained; use the national workflow for QA improvements.

Review scale, missing data, units, and any training/calibration requirements before interpreting results.

Experimental model: use spatial/time holdout validation and independent Bangladesh ground truth; finer pixels do not prove added information.

May contain tutorial-specific assets or imported training features; inspect dependencies before execution.

In [ ]:
# Bangladesh adaptation: 0239_sartram
# Original author: Amirhossein Ahrari; see original attribution below and originals/.
from bdgee.core import initialize, boundaries
from bdgee.notebook import export_outputs, upload_local
initialize()  # Run `earthengine authenticate` first; project is configured in bdgee.
BD_COUNTRY, BD_DISTRICTS = boundaries()
BD_ROI = BD_COUNTRY.geometry()
BD_BBOX = [88.0, 20.5, 92.7, 26.7]  # search only, not analysis boundary
roi = geometry = border = iran = tehran = province = loc = point = BD_ROI
# Run from the repository root; all local outputs are isolated by source.
from pathlib import Path
import os
_REPO_ROOT = Path.cwd()
_OUTPUT_DIR = Path(os.environ.get('BDGEE_OUTPUT_DIR', _REPO_ROOT / 'outputs' / 'soil_drought' / '0239_sartram'))
_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(_OUTPUT_DIR)


In [ ]:
"""00030_sar_soil_moisture.ipynb


Tutorial Code by Amirhossein Ahrari
YouTube: https://www.youtube.com/@amirhosseinahrarigee
Tutorial Video: Google Earth Engine Tutorial-215: SAR Based Soil Moisture Estimation using Sentinel-1 OPERA
This code is part of a tutorial series on Earth Engine programming techniques
presented on the Amirhossein Ahrari YouTube channel. You are free to use and modify
this code for academic and non-academic purposes. Don't forget to subscribe to
the Amirhossein Ahrari channel and follow the videos to support the instructor


"""
import ee
None
None
'# **Study Area**'
import geemap
map = geemap.Map(basemap='SATELLITE')
map
roi = BD_ROI
roi
'# **SAR Image Collection**'
sar = ee.ImageCollection('OPERA/RTC/L2_V1/S1').filterDate('2025', '2026').filterBounds(roi).select('VV', 'VH').filter(ee.Filter.eq('ORBIT_PASS_DIRECTION', 'descending'))
sar
dw = ee.ImageCollection('GOOGLE/DYNAMICWORLD/V1').filterDate('2025', '2026').filterBounds(roi).select('label').mode().eq(6).Not()
'# **to Xarray**'
from xee import helpers
from shapely.geometry import shape
import xarray as xr
sar.first().select('VV').projection()
grid = helpers.fit_geometry(geometry=shape(roi.getInfo()), grid_crs='EPSG:32638', grid_scale=(30, -30))
ds = xr.open_dataset(sar, engine='ee', **grid, chunks='auto')
from dask.diagnostics import ProgressBar
with ProgressBar():
    ds = ds.compute()
ds
mask = xr.open_dataset(dw, engine='ee', **grid)
mask = mask.squeeze('time').drop_vars('time') * 1
mask
'# **Temporal Aggregation**'
ds = ds.sortby('time')
ds_monthly = ds.resample(time='M').mean('time')
ds_smooth = ds_monthly.rolling(time=3, center=True, min_periods=1).mean('time')
ds_mean = ds_smooth.mean(dim='time')
ds_mask = ds_mean.where(mask.label)
rvi = 4 * ds_mask.VH / (ds_mask.VH + ds_mask.VV)
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ds_mask.VV.plot(ax=ax[0], robust=True)
ds_mask.VH.plot(ax=ax[1], robust=True)
rvi.plot(ax=ax[2], robust=True)
plt.tight_layout()
'# **Soil Moisture**'
import numpy as np
x = rvi.values.flatten()
x = x[~np.isnan(x)]
x = x.reshape(-1, 1)
x
y = ds_mask.VH.values.flatten()
y = y[~np.isnan(y)]
y
samples = np.random.choice(len(x), 10000, replace=False)
samples
x_samples = x[samples]
y_samples = y[samples]
from sklearn.linear_model import QuantileRegressor
dry_model = QuantileRegressor(quantile=0.05, alpha=0).fit(x_samples, y_samples)
dry_edge = dry_model.intercept_ + dry_model.coef_[0] * rvi
wet_model = QuantileRegressor(quantile=0.95, alpha=0).fit(x_samples, y_samples)
wet_edge = wet_model.intercept_ + wet_model.coef_[0] * rvi
sm = (ds_mask.VH - dry_edge) / (wet_edge - dry_edge)
sm = sm.clip(0, 1)
sm.plot(robust=True)


In [ ]:

# Review these task plans; task creation does not start exports.
os.chdir(_REPO_ROOT)
bd_tasks = export_outputs(globals(), 'soil_drought', '0239_sartram', scale=1000)
# Set an appropriate native scale above before creating plans.
# for task in bd_tasks: task.start()
# upload_local(_OUTPUT_DIR, 'soil_drought')  # explicit upload; requires Drive OAuth scope
